# Stage 5: Train DistilBERT (4 checkpoints)

**Stage:** 5 (training) · **Runs on:** Kaggle with **GPU T4** and **Internet on** (downloads the
pre-trained model). Locally it runs on CPU but is very slow: train it on Kaggle and copy the four
`distilbert__<dataset>` folders into `outputs/05_models/`; this notebook then skips training and
only makes the predictions.

| | |
|---|---|
| Input | Stage 2 split files (DistilBERT reads raw text; it does not use Stage 4) |
| Output | `distilbert__<dataset>/` × 4 (model folders), `pred__distilbert__<dataset>.csv` × 4 |

For each dataset: fine-tune on the training part with early stopping on its validation part
(the epoch with the lowest validation loss is kept), then predict the validation and test parts
of all four datasets.

**On Kaggle:** attach the `misinfo_utils` utility script and add these notebooks as inputs (**Add Input → Your Work → Notebooks**): `02_split`.

In [1]:
# Setup: import the shared code. Works on Kaggle and locally.
import sys
from pathlib import Path

# Locally, misinfo_utils.py is in <project>/src/. On Kaggle it is an attached Utility Script,
# so this loop finds nothing and the import below still works.
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "misinfo_utils.py").exists():
        sys.path.insert(0, str(folder / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import misinfo_utils as mu

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
paths = mu.setup("05")

misinfo_utils | stage 05 | running on local
  read from : D:\misinfo-project\data\raw, D:\misinfo-project\outputs
  write to  : D:\misinfo-project\outputs\05_models


In [2]:
import os
BERT_MODEL = os.environ.get("MISINFO_BERT_MODEL", "distilbert-base-uncased")
MAX_LEN = 256       # tokens per text (longer articles are cut)
EPOCHS = 3
BATCH = 16
LEARNING_RATE = 2e-5
RETRAIN = False     # False = reuse a saved distilbert__<dataset> folder if one exists

if os.environ.get("MISINFO_QUICK") == "1":   # tiny test runs only
    EPOCHS, MAX_LEN = 1, 64

In [3]:
import time, shutil, inspect
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

device = mu.bert_device()
print("Device:", device, "| GPUs:", torch.cuda.device_count())
if device == "cpu":
    print("No GPU: training will be slow. On Kaggle, turn on 'GPU T4' in the settings panel.")


def to_dataset(tokenizer, df):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "label": df["label"].astype(int).tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}


def training_args(output_dir):
    args = dict(output_dir=output_dir, learning_rate=LEARNING_RATE, lr_scheduler_type="linear",
                weight_decay=0.01, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
                num_train_epochs=EPOCHS, save_strategy="epoch", logging_strategy="epoch",
                load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
                save_total_limit=1, fp16=(device == "cuda"), report_to="none", seed=mu.SEED)
    accepted = inspect.signature(TrainingArguments.__init__).parameters   # names differ across versions
    args["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
    args["warmup_ratio" if "warmup_ratio" in accepted else "warmup_steps"] = 0.1
    return TrainingArguments(**args)

Device: cpu | GPUs: 0
No GPU: training will be slow. On Kaggle, turn on 'GPU T4' in the settings panel.


In [ ]:
texts = {(ds, s): mu.load_split(ds, s) for ds in mu.DATASETS for s in ("val", "test")}
histories, table = {}, {}

for ds in mu.DATASETS:
    t0 = time.time()
    existing = mu.find_dir(mu.bert_dir_name(ds), required=False)
    if existing is not None and (existing / "config.json").exists() and not RETRAIN:
        print(f"{ds}: using the saved model in {existing}")
        tokenizer, model = mu.load_bert(ds)
    else:
        tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(
            BERT_MODEL, num_labels=2, id2label=mu.LABEL_NAMES, label2id={v: k for k, v in mu.LABEL_NAMES.items()})
        checkpoints = str(mu.output_path(f"bert_checkpoints_{ds}", subfolder=f"classification/{ds}"))
        trainer = Trainer(model=model, args=training_args(checkpoints),
                          train_dataset=to_dataset(tokenizer, mu.load_split(ds, "train")),
                          eval_dataset=to_dataset(tokenizer, texts[(ds, "val")]),
                          data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
                          callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
        trainer.train()
        histories[ds] = pd.DataFrame(trainer.state.log_history)
        folder = mu.output_path(mu.bert_dir_name(ds), subfolder=f"classification/{ds}")
        trainer.save_model(str(folder)); tokenizer.save_pretrained(str(folder))
        shutil.rmtree(checkpoints, ignore_errors=True)       # intermediate checkpoints are not needed
        model = trainer.model.eval()

    rows = []
    for (other, split), df in texts.items():
        p = mu.bert_prob_misleading(tokenizer, model, df["text"], max_len=MAX_LEN)
        rows.append(pd.DataFrame({"row_id": df["row_id"], "tested_on": other, "split": split, "prob_misleading": p}))
    preds = pd.concat(rows, ignore_index=True)
    preds.to_csv(mu.output_path(mu.pred_file("distilbert", ds), subfolder=f"classification/{ds}"), index=False)

    table[ds] = {other: f1_score(texts[(other, "val")]["label"].astype(int),
                                 (preds[(preds.tested_on == other) & (preds.split == "val")]["prob_misleading"] >= 0.5).astype(int),
                                 average="macro") for other in mu.DATASETS}
    print(f"{ds}: done in {(time.time() - t0) / 60:.1f} min")

val_f1 = pd.DataFrame(table).T.loc[mu.DATASETS, mu.DATASETS].round(4)
val_f1.index.name, val_f1.columns.name = "trained on", "validated on"
val_f1

liar: using the saved model in D:\misinfo-project\outputs\05_models\classification\liar\distilbert__liar


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

liar: done in 9.5 min


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2999 [00:00<?, ? examples/s]

C:\Users\USER\anaconda3\envs\misinfo\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
if histories:
    fig, axes = plt.subplots(1, len(histories), figsize=(4.5 * len(histories), 3.6), squeeze=False)
    for ax, (ds, h) in zip(axes[0], histories.items()):
        tr = h.dropna(subset=["loss"]) if "loss" in h else pd.DataFrame()
        ev = h.dropna(subset=["eval_loss"]) if "eval_loss" in h else pd.DataFrame()
        if len(tr): ax.plot(tr["epoch"], tr["loss"], "o-", label="training loss")
        if len(ev): ax.plot(ev["epoch"], ev["eval_loss"], "o-", label="validation loss")
        ax.set_title(ds); ax.set_xlabel("epoch"); ax.legend()
    plt.suptitle("DistilBERT loss per dataset (lowest validation loss = kept)")
    plt.tight_layout(); plt.show()